In [1]:
import os 

In [2]:
os.chdir("..//")

In [3]:
os.chdir("..//")

In [4]:
import time 
import math 
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
@dataclass
class GPTConfig:
    block_size:int  = 1024  # ==> block size
    vocab_size:int  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer:int     = 12
    n_head:int      = 12
    n_emb:int       = 768   # ==> embedding dim

In [7]:
import tiktoken 


class DataLoaderLite:
    def __init__(self,B,T):
        self.B  = B     # batch 
        self.T  = T     # sequence length 
        with open("data\gpt_train.txt","r") as f:
            text    = f.read()
        enc         = tiktoken.get_encoding("gpt2")
        tokens      = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f"loaded of {len(self.tokens)} tokens")
        print(f"1 Epoch = {len(self.tokens) // (B*T)} Batches of token")

        self.current_position = 0 

    def next_batch(self):
        B,T     = self.B,self.T
        buff    = self.tokens[self.current_position:self.current_position+B*T+1]
        x       = (buff[:-1]).view(B,T)     # input 
        y       = (buff[1:]).view(B,T)      # target 
        self.current_position   += B*T 
        if self.current_position + (B*T+1)> len(self.tokens):
            self.current_position = 0 
        return x,y

In [8]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection
        self.c_proj.NANOGPT_SCALE_INIT = 1 # flaging this instance -->> Intitialization method <<--  

        self.n_head = config.n_head
        self.n_emb  = config.n_emb
        self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        ## Attention 
        atten   = (q@k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        atten   = atten.masked_fill(self.bias[:,:,:T,:T] == 0,float("-inf"))
        atten   = F.softmax(atten,dim=-1)

        y       = atten @ v                 # (B,nh,T,T) x (B,nh,T,hs) ==> (B,nh,T,hs)
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x
    
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x



In [9]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
      #-----------------------weight sharing scheme ---------------------------------# 
      self.transformer.wte.weight  = self.lm_head.weight
      # ----------------------Parameter Initialization ------------------------------#
      self.apply(self._init_weights) 

  def _init_weights(self,module):
    std = 0.02 
    if isinstance(module,nn.Linear):
      if hasattr(module,"NANOGPT_SCALE_INIT"):
        std *= (2* self.config.n_layer) ** -0.5  # In a block there is 2 residual connection per layer, so total n_layer * 2 residual connection total 
      # (2* self.config.n_layer) ** -0.5 ==> this means 1 / sqrt(total residual connection)
      torch.nn.init.normal_(module.weight,mean = 0.0,std = std)  # weight initialization function 
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias) 
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std = std) 

  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

In [ ]:
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

train_loader = DataLoaderLite(B=4,T=1024)   # 

# https://docs.pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html
torch.set_float32_matmul_precision('high')   # float32 (FP32) >> tensorFloat32 (TF32)  
              


loaded of 338025 tokens
1 Epoch = 82 Batches of token


- ![alt text](<precision.png>)

## Automatic Mixed Precision

- `torch.cuda.amp` provides convenience methods for mixed precision, 
- Where some operations use the `torch.float32` (float) datatype and other operations use `torch.float16` (half). Some operation, like **linear layers** and **convolutions**, are much faster in `float16 or bfloat16`. 
- Other operation, like **reductions**, often require the dynamic range of `float32`.
- Mixed precision tries to match each operation to its appropriate datatype, which can reduce your network’s runtime and memory footprint.



- `https://docs.pytorch.org/docs/stable/amp.html#torch.autocast` refer this website

In [12]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

In [ ]:
## Optimizer 
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    ## ------------------------- Autocasting -----------------------------------## 
    with torch.autocast(device_type="cuda",dtype=torch.bfloat16): 
        logits,loss = model(x,y)
        import code; code.interact(local=locals())

    loss.backward()
    optimizer.step()
    torch.cuda.synchronize() 
    t1                  = time.time()
    dt                  = (t1 - t0) * 1000  # convert seconds to milliseconds
    token_per_second    = (train_loader.B * train_loader.T) / (t1 - t0) 
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms, tok/sec: {token_per_second:.4f}",)

Python 3.9.19 | packaged by conda-forge | (main, Mar 20 2024, 12:38:46) [MSC v.1929 64 bit (AMD64)] on win32
Type "help", "copyright", "credits" or "license" for more information.
(InteractiveConsole)


torch.bfloat16

- PyTorch schedules operations on the GPU asynchronously — the CPU tells the GPU what to do and moves on.
- But the GPU might still be running those operations.
- So when you call .synchronize(), you're telling the CPU:
    - 🛑 "Wait here until the GPU finishes everything you've told it to do."

In [ ]:
torch.cuda.empty_cache()